In [0]:
import json
# intento leer config externa
config_path = "/Volumes/workspace/config/files/config.json"

try:
    with open(config_path, "r") as f:
        config = json.load(f)

    catalog = config["catalog"]
    schema = config["schema"]
    table = config["table"]

# fallback a widgets (para testing manual)
except Exception as e:
    print(f"Fallback a widgets por error: {e}")
    dbutils.widgets.text("catalog", "workspace")
    dbutils.widgets.text("schema", "silver")
    dbutils.widgets.text("table", "vinos_limpios")

    catalog = dbutils.widgets.get("catalog")
    schema = dbutils.widgets.get("schema")
    table = dbutils.widgets.get("table")

target_path=f"{catalog}.{schema}.{table}"

In [0]:
import pyspark.sql.functions as f
from pyspark.sql.window import Window
dfwhite= spark.read.csv("/databricks-datasets/wine-quality/winequality-white.csv",header=True,sep=";",inferSchema=True
                   )
                   
dfred= spark.read.csv("/databricks-datasets/wine-quality/winequality-red.csv",header=True,sep=";",inferSchema=True
                   )



#Agregamos el valor "tipo" de vino para diferenciar los datasets
dfwhite= dfwhite.withColumn("type",f.lit("white"))
dfred=dfred.withColumn("type",f.lit("red"))

#unimos los 2 datasets de vinos.
df= dfwhite.union(dfred)

#se revisan tipos de datos con columnas y estadisticas
df.printSchema()
display(df.summary())

#redondeamos la columna alcohol a 2 decimales.
df=df.withColumn("alcohol",f.round(df["alcohol"],2))

# 1. Creamos una "ventana" ficticia para ordenar los datos
# Como no nos importa el orden previo, le pasamos un valor constante                   
ventana = Window.orderBy(f.lit("A"))
#utilizamos la ventana para agregar una columna con el id del vino.
df = df.withColumn("id_vino", f.row_number().over(ventana))

#Se renombra las columnas, sacando el espacio para futuros errores.
df = df.toDF(*[c.replace(" ", "_") for c in df.columns])

#Se cuentan los nulos por columna
display(df.select([f.sum(f.col(c).isNull().cast("int")).alias(c) for c in df.columns]))


In [0]:

# Generamos un grafico de barras de calidad
quality_dist = df.groupBy("quality", "type").count()
display(quality_dist)

#  Correlación alcohol vs calidad
display(df.select("alcohol", "quality", "type"))


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# Guardamos como tabla Delta
(df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_path))